In [ ]:
import torch
from botorch.exceptions import InputDataWarning

from bott.problem import OptimizationProblem
from bott.physics_models import simulate_cbed
from bott.optimization import run_one_trial

import warnings
warnings.filterwarnings("ignore", category=InputDataWarning) 
# InputDataWarning: Data (outcome observations) is not standardized (std = tensor([5.3673e-11, 5.0607e-10, 4.8782e-10, 7.7036e-11, 5.1824e-14],
#        dtype=torch.float64), mean = tensor([0.0000e+00, 0.0000e+00, 8.4703e-22, 0.0000e+00, 0.0000e+00],
#        dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
#   check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)

In [ ]:
ground_truth = torch.Tensor(simulate_cbed(5,0.5,17, device_simu='gpu')) # abtem takes "cpu" or "gpu"

In [ ]:
# OptimizationProblem would keep all the tensor on the specified device
problem = OptimizationProblem(ground_truth=ground_truth,
                              output_path='./output', 
                              save_results=True, 
                              reduction_params={'reduction_type':'square', 'reduction_kwargs':{'num_tiles':2}},
                              loss_params={'loss_type':'SSE', 'dp_pow': 1}, 
                              norm_arr=False,
                              dim=3, 
                              bounds=[(1,100), (-20, 20), (-20, 20)],
                              noise_std=0,
                              dtype=torch.float64, 
                              device='cuda'
                              ) # "cpu" or "cuda" for physics simulation

In [ ]:
run_one_trial(problem_name='EI', 
              problem=problem, 
              algo='EI', 
              trial=3, 
              n_init_evals=2, 
              max_iter=10, 
              metrics=['obs_val'],
              objective=None,
              noisy=False,
              dtype=torch.float64,
              device_botorch='cuda'
              )